<a href="https://colab.research.google.com/github/nadezhdina-e/PDA_NadShcher/blob/main/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22%D0%9F%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D0%BA%D0%B0_%D0%A5%D1%80%D0%B0%D0%BD%D0%B5%D0%BD%D0%B8%D0%B5_%D0%B8_%D0%BE%D0%B1%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%BA%D0%B0_%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D1%85_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Обработка данных на Python

При работе с данными нам постоянно нужно проводить с ними различные манипуляции: фильтровать, дополнять, очищать от аномалий и агрегировать. До этого момента мы в основном использовали простые текстовые форматы - например, CSV или JSON. Они отлично справлялись с небольшими задачами, поскольку их легко открыть в блокноте или табличном редакторе и прочитать глазами. Однако с ростом объема данных это преимущество сходит на нет: файл на несколько гигабайт уже не откроешь в обычном редакторе, а работать с ним становится тяжело.

У таких форматов есть серьезные ограничения. Например, CSV абсолютно «плоский» и не умеет выражать связи между сущностями, а чтобы точечно изменить или удалить всего одну строчку, программе приходится считывать весь гигантский файл целиком, вносить изменение в оперативной памяти и перезаписывать файл на диск. Это медленно, неэффективно расходует ресурсы и чревато потерей данных при сбое. Кроме того, если нам нужно связать данные из разных файлов (например, пользователей и их заказы), всю логику поддержания целостности этих связей приходится писать вручную в коде.

Для надежного хранения и быстрой работы с такими объемами нужны специализированные инструменты - **реляционные базы данных (СУБД)**, доступ к которым осуществляется с помощью языка **SQL**.




*Больше про плоские файловые форматы, SQLite, SQLAlchemy: https://realpython.com/python-sqlite-sqlalchemy/*

В рамках этой практики мы построим учебный ETL-пайплайн (Extract, Transform, Load). Чтобы не тратить время на сложный парсинг сайтов, мы упростим шаг Extract и сразу возьмем готовый датасет с Kaggle объемом более 1 млн строк. На нем мы отработаем различные трансформации (Transform), настроим сохранение в хранилище (Load) и проведем бенчмарки - сравним время выполнения и расход памяти для каждого инструмента.

In [ ]:
import pandas as pd

In [ ]:
import kagglehub
path = kagglehub.dataset_download("limjeongeun/synthetic-u-s-e-commerce-dataset-1m-orders")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'synthetic-u-s-e-commerce-dataset-1m-orders' dataset.
Path to dataset files: /kaggle/input/synthetic-u-s-e-commerce-dataset-1m-orders


In [ ]:
import time
import os
import gc
import psutil
from contextlib import contextmanager

@contextmanager
def benchmark(tag: str):
    "Это вспомогательная функция для замера времени выполнения и RAM"
    gc.collect()
    process = psutil.Process(os.getpid())
    mem_before = process.memory_info().rss / (1024 * 1024)
    start_time = time.perf_counter()

    yield

    end_time = time.perf_counter()
    mem_after = process.memory_info().rss / (1024 * 1024)

    elapsed = end_time - start_time
    mem_diff = mem_after - mem_before
    print(f"[{tag}] Время: {elapsed:.3f} сек | RAM: {mem_after:.3f} MB (изменение: {mem_diff:+.3f} MB)")

In [ ]:
df_payments = pd.read_csv(path+'/order_payments.csv')
df_payments

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,720fd9e9-cdeb-4dec-a187-f71586eb085a,1,paypal,1,2084.01
1,c0142972-63fa-4af2-8070-f583ab769847,1,credit_card,6,1406.07
2,11bdf634-2b87-4d37-8d76-be1e7aff8f3b,1,voucher,1,911.30
3,11bdf634-2b87-4d37-8d76-be1e7aff8f3b,2,credit_card,8,600.57
4,d29b58ad-af2a-47b4-9214-b2664fb1fdba,1,credit_card,11,170.73
...,...,...,...,...,...
1149366,2e8839ac-e8eb-463b-8508-06bc3e4c4365,1,credit_card,12,459.61
1149367,00b5e050-0b2d-40cd-be54-4e5fe7bd0cf1,1,credit_card,10,2716.71
1149368,35673ee4-ab74-4d07-bda1-987d28dea3e7,1,credit_card,9,3034.70
1149369,f65ee915-853e-4fe1-81b2-8e1c0e14bdef,1,paypal,1,1014.70


## 1. Pandas + Sqlite

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

# создаем файл SQLite базы данных
engine = create_engine("sqlite:///ecommerce.db")

# 1. считываем данные в память
with benchmark("Pandas: pd.read_csv"):
    df_pd_read_csv = pd.read_csv(path+'/order_payments.csv') # отдельно читаем csv для замера
    print(f"Загружено строк: {len(df_pd_read_csv):,}")

# 2. запись в БД через to_sql
with benchmark("Pandas -> SQLite (to_sql chunksize=10k, multi)"):
    df_payments.to_sql(
        name="payments_pandas",
        con=engine,
        if_exists="replace",  # при первом запуске создаем заново
        index=False
    )

Загружено строк: 1,149,371
[Pandas: pd.read_csv] Время: 4.483 сек | RAM: 1650.785 MB (изменение: +55.066 MB)
[Pandas -> SQLite (to_sql chunksize=10k, multi)] Время: 17.130 сек | RAM: 1654.164 MB (изменение: +3.379 MB)


In [ ]:
with engine.connect() as conn:
    count_before = conn.execute(text("SELECT count(*) FROM payments_pandas")).scalar()
    print(f"Строк до повторной загрузки: {count_before:,}")

# симулируем повторный запуск пайплайна (например, парсер собрал новую порцию данных)
with benchmark("Повторный to_sql"):
    df_payments.to_sql(
        name="payments_pandas",
        con=engine,
        if_exists="append", # наивная попытка догрузить данные
        index=False
    )

with engine.connect() as conn:
    count_after = conn.execute(text("SELECT count(*) FROM payments_pandas")).scalar()
    print(f"Строк после наивного append: {count_after:,} (ПОЯВИЛИСЬ ДУБЛИКАТЫ! Записей вдвое больше!)")

# инспектируем схему таблицы, которую создал Pandas
with engine.connect() as conn:
    schema = conn.execute(text("SELECT sql FROM sqlite_master WHERE name = 'payments_pandas'")).scalar()
    print(f"\nСгенерированная Pandas схема таблицы:\n{schema}")

Строк до повторной загрузки: 1,149,371
[Повторный to_sql] Время: 10.287 сек | RAM: 1660.090 MB (изменение: +5.926 MB)
Строк после наивного append: 2,298,742 (ПОЯВИЛИСЬ ДУБЛИКАТЫ! Записей вдвое больше!)

Сгенерированная Pandas схема таблицы:
CREATE TABLE payments_pandas (
	order_id TEXT, 
	payment_sequential BIGINT, 
	payment_type TEXT, 
	payment_installments BIGINT, 
	payment_value FLOAT
)


`pandas.to_sql()` не умеет создавать `PRIMARY KEY` и `UNIQUE INDEX`, он не гарантирует уникальность и создает примитивную схему без ограничений и проверок уникальности или Null-значений.

Повторный запуск ETL-скрипта с `if_exists='append'` приводит к дублированию данных и искажению всей аналитики.

Типы колонок выводятся эвристически (например, `TEXT` вместо фиксированных `VARCHAR` или `INT`/`FLOAT` вместо точного денежного `NUMERIC`).

In [ ]:
# вычисление в памяти через Pandas
with benchmark("Аналитика 1: Pure Pandas (In-Memory)"):
    res_pandas = (
        df_payments
        .groupby("payment_type")
        .agg(
            total_revenue=("payment_value", "sum"),
            orders_count=("order_id", "count"),
            avg_receipt=("payment_value", "mean")
        )
        .reset_index()
        .sort_values(by="total_revenue", ascending=False)
    )

# делегирование вычислений SQLite
sql_query_1 = """
SELECT
    payment_type,
    SUM(payment_value) AS total_revenue,
    COUNT(order_id) AS orders_count,
    ROUND(AVG(payment_value), 2) AS avg_receipt
FROM payments_pandas
GROUP BY payment_type
ORDER BY total_revenue DESC;
"""

with benchmark("Аналитика 1: SQLite via pd.read_sql_query"):
    res_sql = pd.read_sql_query(sql_query_1, con=engine)

display(res_sql)

[Аналитика 1: Pure Pandas (In-Memory)] Время: 0.169 сек | RAM: 1659.969 MB (изменение: +0.000 MB)
[Аналитика 1: SQLite via pd.read_sql_query] Время: 2.433 сек | RAM: 1659.973 MB (изменение: +0.000 MB)


,payment_type,total_revenue,orders_count,avg_receipt
0,credit_card,1.240441e+09,1072832,1156.23
1,paypal,3.874137e+08,362556,1068.56
2,apple_pay,2.376058e+08,184228,1289.74
3,bank_transfer,1.884859e+08,193532,973.93
4,voucher,1.836146e+08,246300,745.49
5,debit_card,1.770219e+08,182068,972.28
6,boleto,2.978598e+07,57226,520.50


Pandas оказался быстрее SQLite-запроса на агрегации - но это ожидаемо для данного сценария: SQLite - это row-oriented движок (строка хранится целиком), и без индекса по колонке группировки ему приходится читать все строки со всеми полями (Full Table Scan), тогда как pandas работает с уже загруженными в RAM непрерывными NumPy-массивами по столбцам - груповые операции реализованы на Cython/C и хорошо используют кэш процессора. При наличии подходящего индекса или при использовании колоночного движка (например, DuckDB) картина могла бы измениться.

## 2. Sqlalchemy

In [ ]:
from sqlalchemy import String, Integer, Float, create_engine
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column

class Base(DeclarativeBase):
    pass

class OrderPayment(Base): # sqlalchemy представляет таблицу как Python-объект (яркий пример ORM)
    __tablename__ = "order_payments"

    # составной первичный ключ защищает от дублирования записей
    order_id: Mapped[str] = mapped_column(String(32), primary_key=True)
    payment_sequential: Mapped[int] = mapped_column(Integer, primary_key=True)
    payment_type: Mapped[str] = mapped_column(String(20), nullable=False)
    payment_installments: Mapped[int] = mapped_column(Integer, nullable=False)
    payment_value: Mapped[float] = mapped_column(Float, nullable=False)

# инициализация БД и создание таблицы
engine = create_engine("sqlite:///ecommerce.db")

Base.metadata.create_all(engine)

Антипаттерн: Создавать 1 млн объектов модели через `OrderPayment(...)` и вызывать `session.add_all()`. Это потребует 1.5–2 ГБ RAM и займет несколько минут на работу Garbage Collector.

Инженерное решение: Чтение данных чанками, например, по 50 тысяч записей через Pandas и компиляция единого батч-запроса через `conn.execute(stmt, chunk_of_dicts)`. Драйвер sqlite3 выполнит его через нативный C-вызов executemany.

In [ ]:
import pandas as pd
from sqlalchemy import delete, insert

CHUNK_SIZE = 50_000

# очищаем таблицу, чтобы не ловить UNIQUE constraint при повторных запусках
with engine.begin() as conn:
    conn.execute(delete(OrderPayment))

# измеряем чистую скорость пакетной вставки
with benchmark("SQLAlchemy Core: Чистый Batch Insert (1M строк)"):
    reader = pd.read_csv(path + "/order_payments.csv", chunksize=CHUNK_SIZE)

    with engine.begin() as conn:
        stmt = insert(OrderPayment)
        for chunk in reader:
            records = chunk.to_dict(orient="records")
            conn.execute(stmt, records)

[SQLAlchemy Core: Чистый Batch Insert (1M строк)] Время: 18.708 сек | RAM: 1675.918 MB (изменение: +19.922 MB)


Отправляем новые/измененные данные по методу **UPSERT** = **UP**date + in**SERT**

Пусть мы хотим поменять данные в существующей записи (`EXISTING_ORDER_ID`) и вставить новые (`NEW_ORDER_ID`).


*   Обычный `insert()` здесь бы сразу упал на первой же записи (`EXISTING_ORDER_ID`)
*   Обычный `update()` проигнорировал бы новую запись `NEW_ORDER_ID`


Именно **Upsert** решает обе задачи за один проход: обновляет старое и вставляет новое без ошибок нарушения уникальности.

**Upsert** - *идемпотентный* метод.
> Идемпотентность — это свойство объекта или операции, при котором повторное применение действия с теми же параметрами дает тот же результат, что и первое



In [ ]:
import uuid
from sqlalchemy import func, select
from sqlalchemy.dialects.sqlite import insert as sqlite_upsert
from sqlalchemy.orm import Session

# задаем существующий ключ для обновления и динамический для новой строки
EXISTING_ORDER_ID = "d29b58ad-af2a-47b4-9214-b2664fb1fdba"
# при каждом запуске генерируем новый id через uuid4, чтобы запись была точно новой
NEW_ORDER_ID = f"new_order_{uuid.uuid4().hex[:8]}"

# формируем паттерн Upsert
upsert_stmt = sqlite_upsert(OrderPayment)
upsert_stmt = upsert_stmt.on_conflict_do_update(
    index_elements=[OrderPayment.order_id, OrderPayment.payment_sequential],
    set_={
        "payment_type": upsert_stmt.excluded.payment_type,
        "payment_installments": upsert_stmt.excluded.payment_installments,
        "payment_value": upsert_stmt.excluded.payment_value,
    },
)

# фиксируем начальное состояние
with Session(engine) as session:
    orig_record = session.get(OrderPayment, (EXISTING_ORDER_ID, 1))
    initial_val = orig_record.payment_value
    initial_count = session.scalar(
        select(func.count()).select_from(OrderPayment)
    )

# эмулируем догрузку данных
# В батче 2 записи: 1 дубликат (старый ID) и 1 новая (новый ID)
new_val = initial_val + 1000.0  # меняем сумму, чтобы зафиксировать UPDATE

dirty_batch = [
    {
        "order_id": EXISTING_ORDER_ID,
        "payment_sequential": 1,
        "payment_type": "credit_card",
        "payment_installments": 2,
        "payment_value": new_val,  # обновление существующей
    },
    {
        "order_id": NEW_ORDER_ID,
        "payment_sequential": 1,
        "payment_type": "voucher",
        "payment_installments": 1,
        "payment_value": 150.0,  # полностью новая запись
    },
]

# выполняем загрузку
with benchmark("SQLAlchemy: Идемпотентный Upsert"):
    with engine.begin() as conn:
        conn.execute(upsert_stmt, dirty_batch)

# проверяем результаты
with Session(engine) as session:
    updated_record = session.get(OrderPayment, (EXISTING_ORDER_ID, 1))
    new_record = session.get(OrderPayment, (NEW_ORDER_ID, 1))
    total_count = session.scalar(select(func.count()).select_from(OrderPayment))
    diff = total_count - initial_count

    print()
    print(f"--- Результаты догрузки данных ---")
    print(f"Записей в батче:        {len(dirty_batch)} (1 дубликат + 1 новая)")
    print(f"Было строк в базе:      {initial_count:,}")
    print(f"Стало строк в базе:     {total_count:,} (+{diff})")
    print(
        f"\n1. Проверка UPDATE: сумма изменилась с {initial_val:.2f} на {updated_record.payment_value:.2f} руб."
    )
    print(
        f"2. Проверка INSERT: запись {new_record.order_id} успешно создана."
    )
    print(f"\n Таблица выросла ровно на количество новых записей без создания дубликатов")

[SQLAlchemy: Идемпотентный Upsert] Время: 0.012 сек | RAM: 1675.918 MB (изменение: +0.000 MB)

--- Результаты догрузки данных ---
Записей в батче:        2 (1 дубликат + 1 новая)
Было строк в базе:      1,149,371
Стало строк в базе:     1,149,372 (+1)

1. Проверка UPDATE: сумма изменилась с 170.73 на 1170.73 руб.
2. Проверка INSERT: запись new_order_713621d2 успешно создана.

 Таблица выросла ровно на количество новых записей без создания дубликатов


In [ ]:
import pandas as pd
from sqlalchemy import text

# sqlalchemy может выполнять запрос в виде сырой строки
sql_query_1 = """
SELECT
    payment_type,
    SUM(payment_value) AS total_revenue,
    COUNT(order_id) AS orders_count,
    ROUND(AVG(payment_value), 2) AS avg_receipt
FROM order_payments
GROUP BY payment_type
ORDER BY total_revenue DESC;
"""

with benchmark(['SQLAlchemy: Сырой SQL']):
  with engine.connect() as conn:
      result = conn.execute(text(sql_query_1))
      for row in result.mappings():
          print(dict(row))
      print()

{'payment_type': 'credit_card', 'total_revenue': 620221662.9, 'orders_count': 536416, 'avg_receipt': 1156.23}
{'payment_type': 'paypal', 'total_revenue': 193706860.69, 'orders_count': 181278, 'avg_receipt': 1068.56}
{'payment_type': 'apple_pay', 'total_revenue': 118802890.36, 'orders_count': 92114, 'avg_receipt': 1289.74}
{'payment_type': 'bank_transfer', 'total_revenue': 94242927.08, 'orders_count': 96766, 'avg_receipt': 973.93}
{'payment_type': 'voucher', 'total_revenue': 91807453.06, 'orders_count': 123151, 'avg_receipt': 745.49}
{'payment_type': 'debit_card', 'total_revenue': 88510939.59, 'orders_count': 91034, 'avg_receipt': 972.28}
{'payment_type': 'boleto', 'total_revenue': 14892989.17, 'orders_count': 28613, 'avg_receipt': 520.5}

[['SQLAlchemy: Сырой SQL']] Время: 1.164 сек | RAM: 1675.918 MB (изменение: +0.000 MB)


In [ ]:
from sqlalchemy import desc, func, select

# еще можно собрать запрос через модель OrderPayment
with benchmark(['SQLAlchemy: через класс OrderPayment']):
  stmt = (
      select(
          OrderPayment.payment_type,
          func.sum(OrderPayment.payment_value).label("total_revenue"),
          func.count(OrderPayment.order_id).label("orders_count"),
          func.round(func.avg(OrderPayment.payment_value), 2).label(
              "avg_receipt"
          ),
      )
      .group_by(OrderPayment.payment_type)
      .order_by(desc("total_revenue"))
  )

  with engine.connect() as conn:
      rows = conn.execute(stmt).all()
      for row in rows:
          print(
              f"{row.payment_type:<15} | total_revenue: {row.total_revenue:>12.2f} | orders_count: {row.orders_count:>6} | avg_receipt: {row.avg_receipt:>8.2f}"
          )
      print()

credit_card     | total_revenue: 620221662.90 | orders_count: 536416 | avg_receipt:  1156.23
paypal          | total_revenue: 193706860.69 | orders_count: 181278 | avg_receipt:  1068.56
apple_pay       | total_revenue: 118802890.36 | orders_count:  92114 | avg_receipt:  1289.74
bank_transfer   | total_revenue:  94242927.08 | orders_count:  96766 | avg_receipt:   973.93
voucher         | total_revenue:  91807453.06 | orders_count: 123151 | avg_receipt:   745.49
debit_card      | total_revenue:  88510939.59 | orders_count:  91034 | avg_receipt:   972.28
boleto          | total_revenue:  14892989.17 | orders_count:  28613 | avg_receipt:   520.50

[['SQLAlchemy: через класс OrderPayment']] Время: 1.228 сек | RAM: 1675.922 MB (изменение: +0.000 MB)


## 3. Polars

### От хранения к вычислениям

В блоке 2 мы решили задачу надёжного хранения собранных данных: SQLAlchemy гарантирует,
что записи попадут в БД без дублей и с соблюдением схемы (Upsert,
PRIMARY KEY, constraints). Это критично для production-пайплайна, но сам
SQLAlchemy - не аналитический движок! Мы видели в блоке 1, что SQLite
проигрывает in-memory вычислениям на агрегациях без индексов.

Теперь переходим к обратной задаче - быстрым и оптимальным вычислениям над уже
накопленными данными. Здесь полезно использовать **Polars**: он не заменяет
SQLAlchemy (хранить данные транзакционно он не умеет), а берёт то, что
уже лежит в БД или файле, и агрегирует это на порядок быстрее Pandas
за счёт колоночного хранения в памяти и ленивых вычислений.

**Polars** - это сравнительно новый высокопроизводительный аналитический движок, написанный на Rust. Он применяется для тяжелых ETL-трансформаций, очистки миллионов строк и работы с колоночными форматами (Parquet) без перегрузки памяти.

В отличие от **Pandas**, который при вызове `read_csv` сразу аллоцирует память под весь датасет (жадное выполнение / Eager), Polars начинает с создания LazyFrame через `scan_csv`. На этом этапе файл физически не читается в память - Polars лишь валидирует заголовки и строит логический граф вычислений.


*О разработке Polars и секретах производительности: https://www.ritchievink.com/blog/2021/02/28/i-wrote-one-of-the-fastest-dataframe-libraries/*

In [ ]:
import polars as pl

In [ ]:
# загружаем оба датасета в RAM до замеров

df_payments_pd = pd.read_csv(path + "/order_payments.csv")
#df_payments_pl = pl.read_csv(path + "/order_payments.csv") # можем прочитать csv

df_payments_pl = pl.read_database(
    query="SELECT * FROM order_payments",
    connection=engine, # а можем прочитать данные напрямую из БД, которую заполнили в блоке 2
)

# Pandas
with benchmark("Pandas: In-Memory GroupBy 1M rows"):
    res_pd = (
        df_payments_pd
        .groupby("payment_type")
        .agg(
            total_revenue=("payment_value", "sum"),
            orders_count=("order_id", "count"),
            avg_receipt=("payment_value", "mean")
        )
        .reset_index()
        .sort_values(by="total_revenue", ascending=False)
    )

# Polars
with benchmark("Polars: In-Memory GroupBy 1M rows"):
    res_pl = (
        df_payments_pl
        .group_by("payment_type")
        .agg(
            pl.col("payment_value").sum().alias("total_revenue"),
            pl.len().alias("orders_count"),
            pl.col("payment_value").mean().round(2).alias("avg_receipt")
        )
        .sort("total_revenue", descending=True)
    )

[Pandas: In-Memory GroupBy 1M rows] Время: 0.174 сек | RAM: 1735.332 MB (изменение: +0.000 MB)
[Polars: In-Memory GroupBy 1M rows] Время: 0.113 сек | RAM: 1736.059 MB (изменение: +0.727 MB)


In [ ]:
# Pandas (чтение CSV + агрегация)
with benchmark("Pandas: Full Pipeline (read_csv + GroupBy)"):
    df_temp = pd.read_csv(path + "/order_payments.csv")
    res_pd = (
        df_temp
        .groupby("payment_type")
        .agg(
            total_revenue=("payment_value", "sum"),
            orders_count=("order_id", "count"),
            avg_receipt=("payment_value", "mean")
        )
        .reset_index()
        .sort_values(by="total_revenue", ascending=False)
    )

# Polars (scan_csv + оптимизированный .collect())
with benchmark("Polars: Full Pipeline (scan_csv + Lazy collect)"):
    res_pl = (
        pl.scan_csv(path + "/order_payments.csv")
        .group_by("payment_type")
        .agg(
            pl.col("payment_value").sum().alias("total_revenue"),
            pl.len().alias("orders_count"),
            pl.col("payment_value").mean().round(2).alias("avg_receipt")
        )
        .sort("total_revenue", descending=True)
        .collect()
    )

[Pandas: Full Pipeline (read_csv + GroupBy)] Время: 1.919 сек | RAM: 1749.277 MB (изменение: +13.219 MB)
[Polars: Full Pipeline (scan_csv + Lazy collect)] Время: 0.298 сек | RAM: 1758.621 MB (изменение: +9.344 MB)


# Задание для лабораторной работы

1. Собрать данные с веб-ресурса одним из известных способов (не стоит брать готовые датасеты с kaggle)
2. Сохранить их в базу данных SQLite и обеспечить возможность подгружать новые данные
3. Разработать не менее 5 **разных** запросов для агрегации данных. Для выполнения запросов использовать модули pandas и sqlalchemy.

Выбор ресурса и способа сбора данных с него осуществляется студентом самостоятельно и согласовывается с преподавателем до начала выполнения последующих этапов работы. Лабораторная работа выполняется в группах по 2-3 человека. Отчет предоставляется один на группу. Защита каждого студента оценивается индивидуально.

Чтобы открывать БД напрямую и собственными глазами видеть записи полезно использовать https://sqlitebrowser.org/

Пример выполнения работы можно посмотреть здесь: https://github.com/karmarszin/Erarta-Artwork-Scraper

# Задание 1


In [ ]:
import requests
from io import StringIO

url = "https://ru.wikipedia.org/wiki/Список_городов_России"

response = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"}
)

tables = pd.read_html(StringIO(response.text))

In [ ]:
print(f"Найдено таблиц: {len(tables)}")

for i, table in enumerate(tables):
    print(f"Таблица {i}: {table.shape}")

Найдено таблиц: 9
Таблица 0: (1126, 9)
Таблица 1: (15, 9)
Таблица 2: (86, 2)
Таблица 3: (8, 2)
Таблица 4: (1, 2)
Таблица 5: (4, 2)
Таблица 6: (4, 2)
Таблица 7: (11, 2)
Таблица 8: (2, 2)


In [ ]:
df_cities = tables[0]
df_cities

,№,Герб,Город,Регион,Федеральный округ,Население,Основание или первое упоминание,Статус города[2],Прежние названия
0,1,NaN,Абаза,Хакасия,Сибирский,12 272,1867,1966,"Абаканский Завод, Абаканско-Заводское"
1,2,NaN,Абакан,Хакасия,Сибирский,184 769,1734,1931,Усть-Абаканское (до 1931)
2,3,NaN,Абдулино,Оренбургская область,Приволжский,17 274,1795,1923,NaN
3,4,NaN,Абинск,Краснодарский край,Южный,39 511,1863,1963,Абинское (до 1863); Абинская (до 1962)
4,5,NaN,Агидель,Башкортостан,Приволжский,14 219,1980,1991,NaN
...,...,...,...,...,...,...,...,...,...
1121,1122,NaN,Ярославль,Ярославская область,Центральный,577 279,1010,1071,NaN
1122,1123,NaN,Ярцево,Смоленская область,Центральный,41 452,1610,1926,NaN
1123,1124,NaN,Ясногорск,Тульская область,Центральный,15 269,1578,1958,Лаптево (до 1965)
1124,1125,NaN,Ясный,Оренбургская область,Приволжский,15 471,1961,1979,NaN


Удалим столбец с гербами и переименуем столбцы, чтобы дальшле было удобнее.


In [ ]:
df_cities = df_cities.drop(columns=["Герб"])

df_cities.columns = [
    "id",
    "city",
    "region",
    "federal_district",
    "population",
    "foundation_year",
    "city_status_year",
    "former_names"
]

df_cities

,id,city,region,federal_district,population,foundation_year,city_status_year,former_names
0,1,Абаза,Хакасия,Сибирский,12 272,1867,1966,"Абаканский Завод, Абаканско-Заводское"
1,2,Абакан,Хакасия,Сибирский,184 769,1734,1931,Усть-Абаканское (до 1931)
2,3,Абдулино,Оренбургская область,Приволжский,17 274,1795,1923,NaN
3,4,Абинск,Краснодарский край,Южный,39 511,1863,1963,Абинское (до 1863); Абинская (до 1962)
4,5,Агидель,Башкортостан,Приволжский,14 219,1980,1991,NaN
...,...,...,...,...,...,...,...,...
1121,1122,Ярославль,Ярославская область,Центральный,577 279,1010,1071,NaN
1122,1123,Ярцево,Смоленская область,Центральный,41 452,1610,1926,NaN
1123,1124,Ясногорск,Тульская область,Центральный,15 269,1578,1958,Лаптево (до 1965)
1124,1125,Ясный,Оренбургская область,Приволжский,15 471,1961,1979,NaN


# Задание 2

In [ ]:
df_cities.dtypes

,0
id,int64
city,object
region,object
federal_district,object
population,object
foundation_year,object
city_status_year,object
former_names,object


Население не число.

В севастополе и Москве в столбце население были сноски, так что чтобы от них избавиться и превратить все в числа, делаем это:



In [ ]:
df_cities["population"] = (
    df_cities["population"]
    .astype(str)
    .str.replace(r"\[\d+\]", "", regex=True) # убираем скобки сноски
    .str.replace(" ", "", regex=False) # убираем пробелы
)

df_cities["population"] = pd.to_numeric(
    df_cities["population"],
    errors="coerce"
).astype("int64")

df_cities.dtypes

,0
id,int64
city,object
region,object
federal_district,object
population,int64
foundation_year,object
city_status_year,object
former_names,object


Создадим SQLite-базу данных и опишем структуру таблицы cities с помощью SQLAlchemy. В качестве первичного ключа используем id, чтобы обеспечить уникальность записей.

In [ ]:
from sqlalchemy import String, Integer, create_engine
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column

class Base(DeclarativeBase):
    pass

class City(Base):
    __tablename__ = "cities"

    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    city: Mapped[str] = mapped_column(String, nullable=False)
    region: Mapped[str] = mapped_column(String, nullable=False)
    federal_district: Mapped[str] = mapped_column(String, nullable=False)
    population: Mapped[int] = mapped_column(Integer, nullable=False)
    foundation_year: Mapped[str] = mapped_column(String, nullable=False)
    city_status_year: Mapped[str | None] = mapped_column(String, nullable=True)
    former_names: Mapped[str | None] = mapped_column(String, nullable=True)

engine = create_engine("sqlite:///cities.db")

Base.metadata.create_all(engine)

Загрузим собранные данные в таблицу cities.

In [ ]:
from sqlalchemy import delete, insert

CHUNK_SIZE = 50_000

# очищаем таблицу перед загрузкой
with engine.begin() as conn:
    conn.execute(delete(City))

# пакетная загрузка данных
with benchmark("SQLAlchemy Core: Batch Insert"):
    reader = [df_cities]

    with engine.begin() as conn:
        stmt = insert(City)
        for chunk in reader:
            records = chunk.to_dict(orient="records")
            conn.execute(stmt, records)

[SQLAlchemy Core: Batch Insert] Время: 0.032 сек | RAM: 1758.738 MB (изменение: +0.000 MB)


Проверим количество записей после загрузки данных в базу.

In [ ]:
from sqlalchemy import text

with engine.connect() as conn:
    count = conn.execute(
        text("SELECT COUNT(*) FROM cities")
    ).scalar()

print(f"Количество записей в таблице cities: {count:,}")

Количество записей в таблице cities: 1,126


Для возможности повторной загрузки и обновления данных используем UPSERT: существующие записи будут обновляться по первичному ключу id, а новые — добавляться в таблицу.

In [ ]:
from sqlalchemy.dialects.sqlite import insert as sqlite_upsert

upsert_stmt = sqlite_upsert(City)

upsert_stmt = upsert_stmt.on_conflict_do_update(
    index_elements=[City.id],
    set_={
        "city": upsert_stmt.excluded.city,
        "region": upsert_stmt.excluded.region,
        "federal_district": upsert_stmt.excluded.federal_district,
        "population": upsert_stmt.excluded.population,
        "foundation_year": upsert_stmt.excluded.foundation_year,
        "city_status_year": upsert_stmt.excluded.city_status_year,
        "former_names": upsert_stmt.excluded.former_names
    }
)

Проверим работу UPSERT на существующей и новой записи.

In [ ]:
from sqlalchemy import select, func

# количество записей до UPSERT
with engine.connect() as conn:
    count_before = conn.execute(
        select(func.count()).select_from(City)
    ).scalar()

# сохраняем исходное население города с id = 1
original_population = int(df_cities.loc[df_cities["id"] == 1, "population"].iloc[0])

# тест: одна существующая запись и одна новая
test_batch = [
    {
        "id": 1,
        "city": df_cities.loc[df_cities["id"] == 1, "city"].iloc[0],
        "region": df_cities.loc[df_cities["id"] == 1, "region"].iloc[0],
        "federal_district": df_cities.loc[df_cities["id"] == 1, "federal_district"].iloc[0],
        "population": original_population + 1000,
        "foundation_year": df_cities.loc[df_cities["id"] == 1, "foundation_year"].iloc[0],
        "city_status_year": df_cities.loc[df_cities["id"] == 1, "city_status_year"].iloc[0],
        "former_names": df_cities.loc[df_cities["id"] == 1, "former_names"].iloc[0]
    },
    {
        "id": 1127,
        "city": "Тестовый город",
        "region": "Тестовый регион",
        "federal_district": "Тестовый округ",
        "population": 100000,
        "foundation_year": "2026",
        "city_status_year": "2026",
        "former_names": None
    }
]

with engine.begin() as conn:
    conn.execute(upsert_stmt, test_batch)

In [ ]:
with engine.connect() as conn:
    count_after = conn.execute(
        select(func.count()).select_from(City)
    ).scalar()

    updated_population = conn.execute(
        select(City.population).where(City.id == 1)
    ).scalar()

    new_city = conn.execute(
        select(City.city).where(City.id == 1127)
    ).scalar()

print(f"Было строк в базе: {count_before}")
print(f"Стало строк в базе: {count_after}")
print(f"Изменение населения: {original_population} - {updated_population}")
print(f"Новая запись: {new_city}")

Было строк в базе: 1126
Стало строк в базе: 1127
Изменение населения: 12272 - 13272
Новая запись: Тестовый город


Работает, теперь вернем все к изначальному виду.

In [ ]:
from sqlalchemy import delete, update

with engine.begin() as conn:
    # удаляем тестовый город
    conn.execute(
        delete(City).where(City.id == 1127)
    )

    # возвращаем исходное население
    conn.execute(
        update(City)
        .where(City.id == 1)
        .values(population=original_population)
    )

In [ ]:
with engine.connect() as conn:
    final_count = conn.execute(
        select(func.count()).select_from(City)
    ).scalar()

    final_population = conn.execute(
        select(City.population).where(City.id == 1)
    ).scalar()

print(f"Количество записей после очистки: {final_count}")
print(f"Население восстановлено: {final_population == original_population}")

Количество записей после очистки: 1126
Население восстановлено: True


# Задание 3

Для начала готовим данные: парсим ДБ, сохраняем ее копию для душевного равновесия, заготавливаем БД для работы с Polars и Pandas - чтобы продемонстрировать преимущества первого повторно.

In [ ]:
from sqlalchemy import select, func, desc, case
from copy import deepcopy
df_wiki_cities = deepcopy(df_cities)
df_wiki_cities # в качестве страховочного хранилища + для тестирования на прямом извлечении из

,id,city,region,federal_district,population,foundation_year,city_status_year,former_names
0,1,Абаза,Хакасия,Сибирский,12272,1867,1966,"Абаканский Завод, Абаканско-Заводское"
1,2,Абакан,Хакасия,Сибирский,184769,1734,1931,Усть-Абаканское (до 1931)
2,3,Абдулино,Оренбургская область,Приволжский,17274,1795,1923,NaN
3,4,Абинск,Краснодарский край,Южный,39511,1863,1963,Абинское (до 1863); Абинская (до 1962)
4,5,Агидель,Башкортостан,Приволжский,14219,1980,1991,NaN
...,...,...,...,...,...,...,...,...
1121,1122,Ярославль,Ярославская область,Центральный,577279,1010,1071,NaN
1122,1123,Ярцево,Смоленская область,Центральный,41452,1610,1926,NaN
1123,1124,Ясногорск,Тульская область,Центральный,15269,1578,1958,Лаптево (до 1965)
1124,1125,Ясный,Оренбургская область,Приволжский,15471,1961,1979,NaN


In [ ]:
# загружаем базы для запросов. сначала поларс, он будет основным инструментом:
query = "SELECT * FROM cities"
df_cities_pl = pl.read_database(query, engine)
# загружаем базу для пандас
with engine.connect() as conn:
    df_cities_pd = pd.read_sql(text(query), conn)

print(f"Polars DataFrame: {df_cities_pl.shape}")
print(f"Pandas DataFrame: {df_cities_pd.shape}")

Polars DataFrame: (1126, 8)
Pandas DataFrame: (1126, 8)


В виде первого агрегирующего запроса посчитаем население в разных федеральных округах. Посчитаем общее население округа, среднее значение по городам округа, медианное значение и проверим, все ли города таблицы были просчитаны.

In [ ]:
# попробуем провести аналитику по фед округам. посчитаем общее население, составим рейтинг
# ради эксперимента напишем пару запросов в нескольких способах

# squalchemy
with benchmark("SQLAlchemy + Pandas: Население и количество городов по федеральным округам"):
    stmt1 = select(
        City.federal_district.label("Федеральный_округ"),
        func.count(City.id).label("Количество_городов"),
        func.sum(City.population).label("Общее_население")
    ).group_by(City.federal_district).order_by(desc("Общее_население"))

    df_q1 = pd.read_sql(stmt1, engine)

print("squalchemy: Население и количество городов по федеральным округам:")
print(df_q1.to_string(index=False))

# Polars
with benchmark("Polars: GroupBy federal_district"):
    res_pl_1 = (
        df_cities_pl
        .group_by("federal_district")
        .agg(
            pl.col("id").count().alias("Количество_городов"),
            pl.col("population").sum().alias("Общее_население")
        )
        .sort("Общее_население", descending=True)
    )
    # Статистические расчеты по стране
    total_population_pl = df_cities_pl["population"].sum()
    mean_population_pl = df_cities_pl["population"].mean()
    median_population_pl = df_cities_pl["population"].median()
    std_population_pl = df_cities_pl["population"].std()

    # Добавляем долю от общего населения
    res_pl_1 = res_pl_1.with_columns(
        (pl.col("Общее_население") * 100.0 / total_population_pl)
        .round(2)
        .alias("Доля_населения_%")
    )

print("\n")
print("POLARS: Население и количество городов по федеральным округам")
print(res_pl_1)

print("\n")
print("СТАТИСТИКА ПО СТРАНЕ (Polars):")
print(f"Общее население страны:    {total_population_pl:>15,.0f} чел.")
print(f"Среднее население города:  {mean_population_pl:>15,.0f} чел.")
print(f"Медианное население:       {median_population_pl:>15,.0f} чел.")
print(f"Стандартное отклонение:    {std_population_pl:>15,.0f} чел.")
print(f"Всего городов:             {len(df_cities_pl):>15,d}")
print("\n")
# Pandas
with benchmark("Pandas: GroupBy federal_district"):
    res_pd_1 = (
        df_cities_pd
        .groupby("federal_district")
        .agg(
            Количество_городов=("id", "count"),
            Общее_население=("population", "sum")
        )
        .reset_index()
        .sort_values(by="Общее_население", ascending=False)
    )
# Статистические расчеты по стране
    total_population_pd = df_cities_pd["population"].sum()
    mean_population_pd = df_cities_pd["population"].mean()
    median_population_pd = df_cities_pd["population"].median()
    std_population_pd = df_cities_pd["population"].std()

    # Добавляем долю от общего населения
    res_pd_1["Доля_населения_%"] = (
        res_pd_1["Общее_население"] * 100.0 / total_population_pd
    ).round(2)
print("PANDAS: Население и количество городов по федеральным округам")
print(res_pd_1.to_string(index=False))
print("\n\n")
print("СТАТИСТИКА ПО СТРАНЕ (Pandas):")
print(f"Общее урбанистическое население страны:    {total_population_pd:>15,.0f} чел.")
print(f"Среднее население города:  {mean_population_pd:>15,.0f} чел.")
print(f"Медианное население:       {median_population_pd:>15,.0f} чел.")
print(f"Стандартное отклонение:    {std_population_pd:>15,.0f} чел.")
print(f"Всего городов:             {len(df_cities_pd):>15,d}")
print("\n\n")



[SQLAlchemy + Pandas: Население и количество городов по федеральным округам] Время: 0.004 сек | RAM: 1758.742 MB (изменение: +0.000 MB)
squalchemy: Население и количество городов по федеральным округам:
Федеральный_округ  Количество_городов  Общее_население
      Центральный                 304         31127120
      Приволжский                 200         19491056
        Сибирский                 114         11553913
  Северо-Западный                 152         11265182
            Южный                  96         10140474
        Уральский                 115          9617235
  Дальневосточный                  82          5053473
Северо-Кавказский                  63          4931127
[Polars: GroupBy federal_district] Время: 0.004 сек | RAM: 1758.742 MB (изменение: +0.000 MB)


POLARS: Население и количество городов по федеральным округам
shape: (8, 4)
┌───────────────────┬────────────────────┬─────────────────┬──────────────────┐
│ federal_district  ┆ Количество_городов ┆ Общее_н

В качестве второго запроса посчитаем топ-5 регионов с наибольшим числом городов.

In [ ]:
# squalchemy
with benchmark("Топ-5 регионов по количеству городов:"):
    stmt2 = select(
        City.region.label("Регион"),
        func.count(City.id).label("Количество_городов")
    ).group_by(City.region).order_by(desc("Количество_городов")).limit(5)

    df_q2 = pd.read_sql(stmt2, engine)
print(df_q2.to_string(index=False))
print("\n")

# Polars
with benchmark("Polars: Топ-5 регионов по числу городов"):
    res_pl_2 = (
        df_cities_pl
        .group_by("region")
        .agg(pl.col("id").count().alias("Количество_городов"))
        .sort("Количество_городов", descending=True)
        .head(5)
    )
print(res_pl_2)
print("\n")

# Pandas
with benchmark("Pandas: Топ-5 регионов по числу городов"):
    res_pd_2 = (
        df_cities_pd
        .groupby("region")
        .agg(Количество_городов=("id", "count"))
        .reset_index()
        .sort_values(by="Количество_городов", ascending=False)
        .head(5)
    )
print(res_pd_2)


[Топ-5 регионов по количеству городов:] Время: 0.005 сек | RAM: 1758.750 MB (изменение: +0.000 MB)
               Регион  Количество_городов
   Московская область                  74
 Свердловская область                  47
Ленинградская область                  37
  Челябинская область                  30
Нижегородская область                  28


[Polars: Топ-5 регионов по числу городов] Время: 0.001 сек | RAM: 1758.750 MB (изменение: +0.000 MB)
shape: (5, 2)
┌───────────────────────┬────────────────────┐
│ region                ┆ Количество_городов │
│ ---                   ┆ ---                │
│ str                   ┆ u32                │
╞═══════════════════════╪════════════════════╡
│ Московская область    ┆ 74                 │
│ Свердловская область  ┆ 47                 │
│ Ленинградская область ┆ 37                 │
│ Челябинская область   ┆ 30                 │
│ Нижегородская область ┆ 28                 │
└───────────────────────┴────────────────────┘


[Pandas: Топ-

Ранее в задании уже рассматривались преимущества Polars над Pandas. Несмотря на больший расход памяти, он работает быстрее и действует оптимальнее. Используем оптимизированный код для Polars и сравним с классическим вариантом.

Третий запрос - распределение городов по десятилетиям основания

In [ ]:
# Pandas (чтение из SQLite + агрегация)
with benchmark("Pandas: Full Pipeline (read_sql + GroupBy foundation decade)"):
    res_pd_3 = (
        df_cities_pd
        .assign(Период_основания=lambda x: x["foundation_year"].astype(str).str[:3] + "0-е годы")
        .groupby("Период_основания")
        .agg(Количество_городов=("id", "count"))
        .reset_index()
        .sort_values(by="Период_основания")
    )
print(f"{res_pd_3} \n")

# Polars Full Pipeline
with benchmark("Polars: GroupBy foundation decade"):
    res_pl_3 = (
        df_cities_pl
        .with_columns(
            (pl.col("foundation_year").cast(pl.Utf8).str.slice(0, 3) + pl.lit("0-е годы"))
            .alias("Период_основания")
        )
        .group_by("Период_основания")
        .agg(
            pl.col("id").count().alias("Количество_городов")
        )
        .sort("Период_основания")
    )
print(f"{res_pl_3} \n")

# здесь пробовали применить pl.scan_database, поскольку читаем с БД, а не с CSV, но наша версия Polars не поддерживает такой функционал, а установкой доп зависимостей заниматься не хочется
# # Polars (scan_database + ленивый collect)
# with benchmark("Polars: Full Pipeline (scan_database + Lazy collect)"):
#     res_pl_3_2 = (
#         pl.scan_database("SELECT * FROM cities", engine=engine)
#         .with_columns(
#             (pl.col("foundation_year").cast(pl.Utf8).str.slice(0, 3) + pl.lit("0-е годы"))
#             .alias("Период_основания")
#         )
#         .group_by("Период_основания")
#         .agg(
#             pl.len().alias("Количество_городов")
#         )
#         .sort("Период_основания")
#         .collect()
#     )

# Polars (read_database + lazy collect)
with benchmark("Polars: Full Pipeline (read_database + Lazy collect)"):
    res_pl_3_2 = (
        df_cities_pl
        .lazy()  # Переводим в ленивый режим для оптимизации
        .with_columns(
            (pl.col("foundation_year").cast(pl.Utf8).str.slice(0, 3) + pl.lit("0-е годы"))
            .alias("Период_основания")
        )
        .group_by("Период_основания")
        .agg(
            pl.len().alias("Количество_городов")
        )
        .sort("Период_основания")
        .collect()
    )
print(res_pl_3_2)


[Pandas: Full Pipeline (read_sql + GroupBy foundation decade)] Время: 0.008 сек | RAM: 1758.750 MB (изменение: +0.000 MB)
    Период_основания  Количество_городов
0        1000-е годы                   1
1        1010-е годы                   1
2        1030-е годы                   1
3        1060-е годы                   1
4        1070-е годы                   2
..               ...                 ...
111      XV 0-е годы                   8
112      XVI0-е годы                  42
113      XX 0-е годы                   1
114      кон0-е годы                   1
115      сер0-е годы                   1

[116 rows x 2 columns] 

[Polars: GroupBy foundation decade] Время: 0.004 сек | RAM: 1758.750 MB (изменение: +0.000 MB)
shape: (116, 2)
┌──────────────────┬────────────────────┐
│ Период_основания ┆ Количество_городов │
│ ---              ┆ ---                │
│ str              ┆ u32                │
╞══════════════════╪════════════════════╡
│ 1000-е годы      ┆ 1                 

Судя по полученным benchmark не имеет смысла использовать модифицированный Polars. Интереса ради следующий запрос выполним с помощью перевода БД в формат CSV.

Четвертый запрос - средний возраст городов по федеральным округам

In [ ]:
# Polars (read_database + lazy collect)
csv_path = "/tmp/cities.csv"
df_cities_pl.write_csv(csv_path)

with benchmark("Polars: Full Pipeline (scan_csv + Lazy collect)"):
    res_pl_4 = (
        pl.scan_csv(csv_path)
        .with_columns(
            pl.col("foundation_year").cast(pl.Utf8).str.to_integer(strict=False)
            .alias("year_num")
        )
        # Вычисляем возраст города
        .with_columns(
            (pl.lit(2026) - pl.col("year_num")).alias("age")
        )
        # Отбрасываем строки с некорректным годом
        .filter(pl.col("age").is_not_null())
        .group_by("federal_district")
        .agg(
            # Средний возраст
            pl.col("age").mean().round(1).alias("Средний_возраст_лет"),
            # Самый старый город: сортируем по возрастанию
            pl.col("city").sort_by("year_num", descending=False).first()
                .alias("Самый_старый_город"),
            # Самый молодой город
            pl.col("city").sort_by("year_num", descending=True).first()
                .alias("Самый_молодой_город"),

            pl.col("year_num").min().alias("Минимальный_год"),
            pl.col("year_num").max().alias("Максимальный_год"),
        )
        .sort("Средний_возраст_лет", descending=True)
        .collect()
    )

print("POLARS: Средний возраст городов по федеральным округам")
print(res_pl_4)

[Polars: Full Pipeline (scan_csv + Lazy collect)] Время: 0.014 сек | RAM: 1758.781 MB (изменение: +0.031 MB)
POLARS: Средний возраст городов по федеральным округам
shape: (8, 6)
┌────────────────┬────────────────┬────────────────┬───────────────┬───────────────┬───────────────┐
│ federal_distri ┆ Средний_возрас ┆ Самый_старый_г ┆ Самый_молодой ┆ Минимальный_г ┆ Максимальный_ │
│ ct             ┆ т_лет          ┆ ород           ┆ _город        ┆ од            ┆ год           │
│ ---            ┆ ---            ┆ ---            ┆ ---           ┆ ---           ┆ ---           │
│ str            ┆ f64            ┆ str            ┆ str           ┆ i64           ┆ i64           │
╞════════════════╪════════════════╪════════════════╪═══════════════╪═══════════════╪═══════════════╡
│ Центральный    ┆ 452.9          ┆ Муром          ┆ Десногорск    ┆ 862           ┆ 1974          │
│ Северо-Западны ┆ 404.0          ┆ Белозерск      ┆ Костомукша    ┆ 862           ┆ 1977          │
│ й           

Никакого впечатляюще быстрого результата мы не увидели, поэтому пятый запрос проведем на классическом Polars. Пускай пятый запрос - найти наиболее урбанизированные регионы - мало городов и много населения.

In [ ]:
with benchmark("Polars: Full Pipeline (lazy)"): # на CSV
    res_pl_5 = (
        pl.scan_csv(csv_path)
        .group_by("region")
        .agg(
            pl.col("population").sum().alias("Суммарное_население"),
            pl.len().alias("Количество_городов")
        )
        .filter(
            (pl.col("Суммарное_население") > 1_000_000) &
            (pl.col("Количество_городов") < 15)
        )
        .sort("Суммарное_население", descending=True)
        .collect()
    )
print(f"{res_pl_5}\n")

# Polars Full Pipeline
with benchmark("Polars: GroupBy Population/Number of cities proportion"): # на DF
    res_pl_5_2 = (
        df_cities_pl
        .lazy()
        .group_by("region")
        .agg(
            pl.col("population").sum().alias("Суммарное_население"),
            pl.len().alias("Количество_городов")
        )
        .filter(
            (pl.col("Суммарное_население") > 1_000_000) &
            (pl.col("Количество_городов") < 15)
        )
        .sort("Суммарное_население", descending=True)
        .collect()
    )

print(f"{res_pl_5_2}")


[Polars: Full Pipeline (lazy)] Время: 0.009 сек | RAM: 1758.797 MB (изменение: +0.008 MB)
shape: (10, 3)
┌───────────────────────┬─────────────────────┬────────────────────┐
│ region                ┆ Суммарное_население ┆ Количество_городов │
│ ---                   ┆ ---                 ┆ ---                │
│ str                   ┆ i64                 ┆ u32                │
╞═══════════════════════╪═════════════════════╪════════════════════╡
│ Москва                ┆ 13010112            ┆ 1                  │
│ Санкт-Петербург       ┆ 5601911             ┆ 1                  │
│ Самарская область     ┆ 2392558             ┆ 11                 │
│ Новосибирская область ┆ 2032881             ┆ 14                 │
│ Приморский край       ┆ 1305137             ┆ 12                 │
│ Дагестан              ┆ 1278857             ┆ 10                 │
│ Омская область        ┆ 1214794             ┆ 6                  │
│ Алтайский край        ┆ 1192230             ┆ 12                 

Вот на этом эксперименте мы увидели, что при сканировании csv результат может быть лучше, чем при сканировании датафрейма